# Centauro-Lite — varredura de experimentos

Treina varias versoes do modelo, uma por configuracao, e compara todas na mesma tabela.

A primeira rodada deu 0,9240 sem treino contra 0,6421 treinado — 30,5% de queda com
apenas 49 passos e 1 epoca. Isso e pouquissimo treino, e a pergunta obvia e quanto ainda
sobra na mesa. Este notebook responde variando **uma coisa por vez**: numa tabela onde
duas coisas mudaram juntas, nenhuma diferenca pode ser atribuida a nenhuma das duas.

---

## Como rodar (leia antes)

**Nao use Run All.** Uma sessao interativa do Kaggle e derrubada depois de **1 hora sem
interacao**, e esta varredura leva horas — fechar a aba mataria o treino.

Use **Save Version -> Save & Run All (Commit)**. Isso executa o notebook no servidor do
Kaggle, sem depender do navegador. Voce fecha tudo e volta depois; o resultado fica em
**Versions**.

**Configuracoes no painel da direita:** Accelerator `GPU T4 x2` · Internet `On` ·
Persistence `Files only`.

## Por que dois grupos

Uma sessao em lote tambem morre em **12 horas**, e a varredura inteira passa disso. Uma
execucao cortada no teto perde o que nao terminou, entao ela e dividida:

| grupo | rodadas | estimativa |
|---|---|---|
| `group_a` | `epochs3`, `rank32`, `lr1e4`, `attention_only`, `best_guess` + Minitaur | ~7h |
| `group_b` | `epochs5`, `rank16`, `rank64`, `lr2e4` | ~7h |

O grupo A tem as rodadas com maior chance de mudar a conclusao. Se ele ja responder o
que interessa, o B vira opcional.

**Rode o A primeiro.** Depois, para o B, veja a secao 6 — ha um passo que faz os
resultados dos dois aparecerem na mesma tabela.

Repositorio: https://github.com/Mathwesm/tcc_projeto.git

## 1. Escolha o grupo

Mude esta linha entre uma execucao e outra. E a unica coisa que muda entre o grupo A e
o grupo B.

In [ ]:
GROUP = "group_a"  # troque para "group_b" na segunda execucao

## 2. Ambiente

In [ ]:
# Uma T4 x2 expoe duas placas, e o unsloth nao lida bem com as duas ao mesmo tempo.
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

In [ ]:
import os
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "tcc_projeto"

if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/Mathwesm/tcc_projeto.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

os.chdir(REPO_DIR)
os.environ["PYTHONUTF8"] = "1"

# O `pip install -e` registra o pacote via um arquivo em site-packages, e um
# interpretador que ja esta rodando nao releia isso -- o kernel deste notebook so
# enxergaria o pacote se fosse reiniciado. Subprocessos (`!python -m centauro_lite`)
# funcionam porque nascem depois, o que mascarou o problema ate uma celula tentar
# importar direto. Apontar o sys.path e o PYTHONPATH resolve os dois casos.
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC

!pip install -q -e . --no-deps
!pip install -q pydantic pydantic-settings loguru typer pyyaml datasets pandas matplotlib
print("cwd:", Path.cwd())

## 3. Aproveitar resultados de execucoes anteriores

Uma sessao em lote sempre **comeca do zero**: ela nao enxerga o que a execucao anterior
fez. Sem isso, o grupo B produziria uma tabela so com as rodadas dele, e a comparacao com
o grupo A se perderia.

Para juntar os dois, anexe a saida da execucao anterior como entrada, no painel da
direita: **Add Input -> Your Work -> Notebook Output**, e escolha a versao que rodou o
grupo A. A celula abaixo encontra o `eval_results.json` dela sozinha e incorpora.

Na **primeira** execucao nao ha nada para anexar, e a celula simplesmente nao acha nada.
Isso e esperado.

In [ ]:
import json
from pathlib import Path

target = Path("outputs/eval_results.json")
merged = json.loads(target.read_text(encoding="utf-8")) if target.is_file() else {}

found = (
    sorted(Path("/kaggle/input").rglob("eval_results.json"))
    if Path("/kaggle/input").exists()
    else []
)
for path in found:
    previous = json.loads(path.read_text(encoding="utf-8"))
    # Rodadas ja medidas entram; as chaves com "_" sao metadados e sao reescritas
    # pela avaliacao seguinte de qualquer jeito.
    merged.update({k: v for k, v in previous.items() if not k.startswith("_")})
    print("incorporado:", path)

if merged:
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")

print()
print(f"{len([k for k in merged if not k.startswith('_')])} rodadas ja medidas:")
for name in sorted(k for k in merged if not k.startswith("_")):
    print(" ", name)

## 4. O que vai ser testado

Confira antes de gastar horas de GPU.

In [ ]:
from pathlib import Path

from centauro_lite.models.pipeline_config import PipelineConfig

default = PipelineConfig.from_yaml(Path("configs/default.yaml"))
print(f"{'rodada':<18} {'rank':>5} {'epocas':>7} {'lr':>9} {'modulos':>8}")
print("-" * 52)
print(
    f"{'(base ja medida)':<18} {default.model.lora_rank:>5} {default.training.num_epochs:>7.0f}"
    f" {default.training.learning_rate:>9.0e} {len(default.model.target_modules):>8}"
)
for path in sorted(Path(f"configs/sweep/{GROUP}").glob("*.yaml")):
    config = PipelineConfig.from_yaml(path)
    print(
        f"{path.stem:<18} {config.model.lora_rank:>5} {config.training.num_epochs:>7.0f}"
        f" {config.training.learning_rate:>9.0e} {len(config.model.target_modules):>8}"
    )

# Todas as rodadas precisam usar os MESMOS dados e os MESMOS participantes, senao a
# tabela compara coisas diferentes. A suite de testes ja garante isso, mas conferir
# aqui custa nada e o erro sairia caro.
configs = [PipelineConfig.from_yaml(p) for p in Path("configs/sweep").rglob("*.yaml")]
print()
print("fingerprint dos dados: ", {c.data_fingerprint for c in configs} | {default.data_fingerprint})
print(
    "fingerprint do split:  ", {c.split_fingerprint for c in configs} | {default.split_fingerprint}
)

## 5. Rodar

Cada rodada acontece num processo separado. Carregar e descartar modelos quantizados
repetidamente no mesmo processo fragmenta a VRAM, e numa placa de 16 GB e a quarta rodada
que morre; processo novo tambem significa que um erro custa uma linha da tabela, nao a
varredura inteira.

Rodadas ja medidas — inclusive as que vieram da execucao anterior, na secao 3 — sao
puladas.

In [ ]:
!python -m centauro_lite sweep --configs configs/sweep/{GROUP}

## 6. Minitaur-8B (so no grupo A)

O Minitaur e a versao de 8 bilhoes do Centaur, publicada pelos proprios autores com a
mesma receita. E a comparacao que sustenta o trabalho: mesmo split, mesmo codigo, mesma
metrica.

Repare que sao **duas** etapas. O dataset guarda ids de token, e um id pertence a um
vocabulario so — o id 2610 e `You` no Qwen3 e ` askear` no Llama. Avaliar o Minitaur
sobre os dados tokenizados para o Qwen3 faria ele ler ruido, devolver um numero
plausivel e nao levantar erro nenhum. Foi exatamente o que aconteceu na primeira
tentativa. O `prepare` roda de novo com o `configs/minitaur.yaml`: os mesmos
participantes (o fingerprint do split nao muda), com o vocabulario dele.

Ressalva para o texto do TCC: NLL por token nunca e perfeitamente comparavel entre
tokenizadores diferentes, porque cada um corta o texto em pedacos diferentes. Dar a cada
modelo o seu vocabulario e o piso, nao a solucao.

In [ ]:
import json
from pathlib import Path

results_path = Path("outputs/eval_results.json")
results = json.loads(results_path.read_text(encoding="utf-8")) if results_path.is_file() else {}

if "minitaur-8b" in results:
    print("Minitaur ja medido:", results["minitaur-8b"]["nll"])
else:
    !python -m centauro_lite prepare --config configs/minitaur.yaml
    !python -m centauro_lite evaluate --config configs/minitaur.yaml --label "minitaur-8b"

## 7. A tabela

A coluna `vs base` e a queda do NLL contra o modelo sem treino, medida **dentro da mesma
configuracao de dados**. Linhas com fingerprints diferentes aparecem com um aviso: elas
foram medidas sobre dados preparados de forma diferente e nao se comparam diretamente.

In [ ]:
!python -m centauro_lite report

In [ ]:
from IPython.display import Image, display

display(Image("outputs/figures/ablation.png"))
display(Image("outputs/figures/per_experiment.png"))

## 8. Guardar

Se voce rodou via **Save & Run All**, a saida ja fica anexada a versao automaticamente —
nao precisa fazer nada.

Depois me mande o conteudo de `outputs/report.txt`: e a tabela inteira em texto, mais
facil de ler que print de tela.

E se for rodar o grupo B: anote qual versao acabou de rodar, porque e ela que voce vai
anexar em **Add Input -> Your Work -> Notebook Output** na proxima execucao.

In [ ]:
!ls -lh outputs/
!cat outputs/report.txt